# Open Models in Jupyter

Most of the time you use AI through a website or an app. The model lives somewhere else and you never see it.

Here the model runs right in this notebook. You load it, talk to it, and look at exactly what it sees.

## Part 1: Run the Model

We have one model in four sizes. Same family, same training recipe. The only thing that changes is size. By the end of this part you will watch the same questions go from wrong to right as the model gets bigger, and you will see why.

### Find the models

Your instructor already downloaded them. Let's see what we have.

In [ ]:
from pathlib import Path

folder = Path("/home/jovyan/shared/open-models")   # on the hub
if not folder.exists():
    folder = Path("models")                        # on your laptop

for f in sorted(folder.glob("*.gguf")):
    print(f"{f.name:28} {f.stat().st_size / 1e9:.1f} GB")

Each `.gguf` file is a whole model. One file. The number in the name is how many parameters it has, in billions.

### Load one

Start with the smallest.

In [ ]:
import shutil
from llama_cpp import Llama

def load(size):
    return Llama(
        model_path=str(folder / f"Qwen3.5-{size}-Q4_K_M.gguf"),
        n_ctx=4096,                                            # how much text it can look at once
        n_gpu_layers=-1 if shutil.which("nvidia-smi") else 0,  # use the GPU if there is one
        verbose=False,
    )

model = load("0.8B")

### Ask it something

In [ ]:
messages = [
    {"role": "user", "content": "What is a language model? One sentence."},
]

response = model.create_chat_completion(messages=messages, max_tokens=200)
response

That is everything the model sent back. It is just a Python dictionary. The answer is in here:

In [ ]:
print(response["choices"][0]["message"]["content"])

Look at `usage` in the dictionary too. That is how many tokens went in and how many came out.

### What did the model actually see?

We gave it a list. The model does not read lists. Your messages get turned into one long string with special markers first. The **chat template** builds that string, and it comes inside the model file.

In [ ]:
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

template = Jinja2ChatFormatter(
    template=model.metadata["tokenizer.chat_template"],
    eos_token="",
    bos_token="",
)

prompt = template(messages=messages).prompt
print(prompt)

- `<|im_start|>` and `<|im_end|>` mark where each message starts and stops.
- The empty `<think>` block tells this model to skip thinking out loud and just answer. We will come back to this.
- The string stops right where the assistant should start talking. The model's whole job is to keep going.

### Then it becomes numbers

In [ ]:
tokens = model.tokenize(prompt.encode(), add_bos=False, special=True)
print(len(tokens), "tokens")
print(tokens)

In [ ]:
[model.detokenize([t], special=True).decode(errors="replace") for t in tokens]

Every piece of text is a number. The model only ever sees the numbers, and it answers with numbers that get turned back into text.

That token count matches `prompt_tokens` in the response above.

### Send the string yourself

Since the prompt is just a string, we can build it and send it ourselves. That lets us change the template, which we need for thinking below. Same model, same answer, one short function.

In [ ]:
def ask(model, question, think=False, max_tokens=150):
    prompt = template(messages=[{"role": "user", "content": question}], enable_thinking=think).prompt
    out = model.create_completion(prompt, max_tokens=max_tokens, temperature=0, stop=["<|im_end|>"])
    return out["choices"][0]["text"].strip()

ask(model, "What is a language model? One sentence.")

`temperature=0` means it always picks the most likely next token, so you get the same answer every time. More on that in Part 3.

### Can it count letters?

In [ ]:
ask(model, "How many times does the letter r appear in the word strawberry? Just the number.")

Wrong. Look at what the model got.

In [ ]:
for text in ["the word strawberry", "strawberry", "s-t-r-a-w-b-e-r-r-y"]:
    pieces = [model.detokenize([t]).decode() for t in model.tokenize(text.encode(), add_bos=False)]
    print(f"{text:22} {pieces}")

In our question, `strawberry` is **one token**. The model never sees the letters. It is like asking someone how many r's are in a word they have only ever heard out loud. It has to guess.

Spell it out with dashes and every letter is its own token. Now there is something to count.

Where do tokens come from? Try the [Token Lab](https://ds-modules.github.io/open-models-2026/token-lab.html). Watch a vocabulary get built from scratch, one merge at a time.

### Thinking on and off

Remember the empty `<think>` block? Turn thinking on and look at how the end of the prompt changes.

In [ ]:
print(template(messages=messages, enable_thinking=True).prompt)

Now the `<think>` block is open. The model writes its own notes first, closes the block, then answers.

Give it something to think about.

In [ ]:
friday = "What day is three days after Friday?"

print(ask(model, friday))

In [ ]:
print(ask(model, friday, think=True, max_tokens=500))

Watch what it does. It counts, loses track, starts over, and runs out of room before it ever answers. A small model does not know when to stop.

Try the same thing on the 4B.

In [ ]:
bigger = load("4B")

print(ask(bigger, friday, think=True, max_tokens=500))

Thinking is not free. Every one of those words is a token the model had to generate before it answered.

### Up the ladder

Same questions, all four sizes, thinking off. This takes a minute.

In [ ]:
import pandas as pd

del bigger

questions = {
    "World Cup":  "Who won the 2034 FIFA World Cup? One sentence.",
    "Australia":  "What is the capital of Australia? Just the city.",
    "Feathers":   "Which is heavier, a pound of feathers or a pound of bricks? One sentence.",
    "Strawberry": "How many times does the letter r appear in the word strawberry? Just the number.",
    "Friday":     "What day is three days after Friday? Just the day.",
    "Sally":      "Sally has 3 brothers. Each of her brothers has 2 sisters. How many sisters does Sally have? Just the number.",
    "Tutor":      "I'm practicing my times tables. Don't tell me the answer, just give me a hint. What is 7 times 8?",
}

answers = {}
for size in ["0.8B", "2B", "4B", "9B"]:
    m = load(size)
    answers[size] = [ask(m, q) for q in questions.values()]
    del m

pd.set_option("display.max_colwidth", 70)
pd.DataFrame(answers, index=questions.keys())

Go row by row. Where does each one flip from wrong to right?

The small models are not broken. They match the **shape** of the question and give the answer that usually goes with that shape.

- "Who won the World Cup?" usually comes with a team name. So it names a team, for a tournament that has not happened.
- Australia's most famous city is Sydney.
- A word problem with two numbers in it usually means multiply. So Sally gets 6 or 12 sisters.

Bigger models learned finer patterns, so the shape stops fooling them. Look at the tutor row though. "7 times 8" is followed by "56" so often that even the biggest one has trouble keeping it to itself.

### Try it

1. Add your own question to `questions` and run the ladder again. Can you find one that only the 9B gets right?
2. Change the Sally question to say "Think step by step." Does a small model get it now?
3. Turn on thinking for the Sally question with the 2B. Does it help, or does it get lost?